# 04 - Domain-Adaptive Pretraining (DAP)

Takes the CoNLL baseline checkpoint from notebook 03 and continues **masked-language-model
pretraining** on each target domain's unlabeled train text (token sequences only, tags
ignored -- the whole point of DAP is that it needs no labels). One run per target dataset,
producing two checkpoints:

- `models/dap_wnut17/`
- `models/dap_scierc/`


In [ ]:
%pip install -q transformers datasets accelerate seqeval

In [ ]:
import json
import math
import sys
import time
from pathlib import Path

import pandas as pd

if Path('/content').exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

for cand in [Path('/content/drive/MyDrive/AAI590/utils'), Path.cwd(),
             Path.cwd() / 'utils', Path.cwd().parent / 'utils']:
    if (cand / 'ner_common_utils.py').exists():
        sys.path.append(str(cand))
        break
import ner_common_utils as ncu

processed_dir = ncu.resolve_processed_dir()
OUTPUT_ROOT = ncu.resolve_output_root(processed_dir)
print('processed dir:', processed_dir)
print('output root  :', OUTPUT_ROOT)

In [ ]:
SMOKE_TEST = False

# 5 epochs as the starting point -- the perplexity table at the end of the training
# cell is there so we actually check this is sensible instead of guessing.
DAP_EPOCHS = 5 if not SMOKE_TEST else 1
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
MLM_PROBABILITY = 0.15
MAX_LENGTH = 256

suffix = '_smoke' if SMOKE_TEST else ''

baseline_dir = OUTPUT_ROOT / 'models' / 'baseline_conll'
if not baseline_dir.exists():
    baseline_dir = OUTPUT_ROOT / 'models' / 'baseline_conll_smoke'
    print('WARNING: real baseline not found, using the smoke checkpoint:', baseline_dir)
assert baseline_dir.exists(), 'run notebook 03 first'
print('SMOKE_TEST:', SMOKE_TEST, '| starting from', baseline_dir)

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoModelForMaskedLM, AutoTokenizer,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)


def make_mlm_dataset(rows, tokenizer):
    def encode(batch):
        return tokenizer(batch['tokens'], is_split_into_words=True,
                         truncation=True, max_length=MAX_LENGTH)
    ds = Dataset.from_list([{'tokens': r['tokens']} for r in rows])
    return ds.map(encode, batched=True, remove_columns=['tokens'])


dap_info = {}
for ds_name in ncu.TARGET_DATASETS:
    print(f'\n================ DAP on {ds_name} ================')
    train_rows = ncu.load_jsonl(processed_dir / ds_name / f'{ds_name}_train.jsonl')
    val_rows = ncu.load_jsonl(processed_dir / ds_name / f'{ds_name}_validation.jsonl')
    if SMOKE_TEST:
        train_rows, val_rows = train_rows[:200], val_rows[:50]

    tokenizer = AutoTokenizer.from_pretrained(baseline_dir)
    train_ds = make_mlm_dataset(train_rows, tokenizer)
    val_ds = make_mlm_dataset(val_rows, tokenizer)

    ncu.set_seed(42)
    model, loading_info = AutoModelForMaskedLM.from_pretrained(
        baseline_dir, output_loading_info=True)
    print('freshly initialized (expected -- this is the new MLM head):',
          loading_info['missing_keys'])
    print('dropped from checkpoint (expected -- the old NER head):',
          loading_info['unexpected_keys'])

    args = TrainingArguments(
        output_dir=str(OUTPUT_ROOT / 'tmp_trainer' / f'dap_{ds_name}{suffix}'),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=DAP_EPOCHS,
        weight_decay=0.01,
        eval_strategy='epoch',
        logging_strategy='epoch',
        save_strategy='no',
        report_to='none',
        disable_tqdm=True,
        seed=42,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer, mlm_probability=MLM_PROBABILITY),
        processing_class=tokenizer,
    )

    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0

    ppl_rows = [{'epoch': h['epoch'], 'eval_loss': h['eval_loss'],
                 'perplexity': math.exp(h['eval_loss'])}
                for h in trainer.state.log_history if 'eval_loss' in h]
    print(pd.DataFrame(ppl_rows).round(3).to_string(index=False))

    out_dir = OUTPUT_ROOT / 'models' / f'dap_{ds_name}{suffix}'
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f'saved -> {out_dir}  ({train_seconds:.0f}s)')

    dap_info[ds_name] = {
        'checkpoint_dir': str(out_dir),
        'epochs': DAP_EPOCHS,
        'train_sentences': len(train_rows),
        'train_seconds': train_seconds,
        'eval_perplexity_by_epoch': ppl_rows,
    }

    del trainer, model
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

In [ ]:
results_dir = OUTPUT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
out_fp = results_dir / f'dap_training_info{suffix}.json'
with open(out_fp, 'w') as f:
    json.dump(dap_info, f, indent=2)
print('saved ->', out_fp)